# 🔴 Solution: Conv2D Forward (NumPy)

In [ ]:
import numpy as np

In [ ]:
# ✅ SOLUTION

def conv2d_forward(x, w, b, stride=1, padding=0):
    # x: (N, C, H, W);  w: (F, C, KH, KW);  b: (F,)
    N, C, H, W = x.shape
    F, _, KH, KW = w.shape
    s, p = stride, padding

    H_out = (H + 2 * p - KH) // s + 1
    W_out = (W + 2 * p - KW) // s + 1

    # Zero-pad the spatial dims only
    x_pad = np.pad(x, ((0, 0), (0, 0), (p, p), (p, p))) if p > 0 else x

    out = np.zeros((N, F, H_out, W_out), dtype=np.float64)

    # Loop over output positions (few); vectorise over N, C, F
    for i in range(H_out):
        for j in range(W_out):
            patch = x_pad[:, :, i * s:i * s + KH, j * s:j * s + KW]   # (N, C, KH, KW)
            # Contract over C, KH, KW -> (N, F). No kernel flip: cross-correlation.
            out[:, :, i, j] = np.tensordot(patch, w, axes=([1, 2, 3], [1, 2, 3])) + b

    return out

In [ ]:
# Verify
np.random.seed(0)
x = np.random.randn(2, 3, 8, 8)
w = np.random.randn(4, 3, 3, 3)
b = np.random.randn(4)

print("stride=1 padding=0:", conv2d_forward(x, w, b, 1, 0).shape)
print("stride=1 padding=1:", conv2d_forward(x, w, b, 1, 1).shape)
print("stride=2 padding=1:", conv2d_forward(x, w, b, 2, 1).shape)

ident = np.zeros((1, 1, 3, 3)); ident[0, 0, 1, 1] = 1.0
one = np.random.randn(1, 1, 5, 5)
print("identity kernel   :", np.allclose(conv2d_forward(one, ident, np.zeros(1), 1, 1), one))

In [ ]:
from torch_judge import check
check("numpy_conv2d")